In [18]:
from datetime import datetime
import re
import json
import csv
from dataclasses import asdict, dataclass
from abc import ABC, abstractmethod

JSON_PATH = "transactions_dirty.json"
CSV_PATH = "transactions_dirty.csv"

@dataclass
class Operation:
  type: str            # deposit / withdraw / interest
  amount: float
  datetime: datetime
  balance: float       # баланс после операции
  status: str          # success / fail
  
class TransactionLoader(ABC):
  # Читает файл и возвращает грязный список словарей без валидации

  @abstractmethod
  def load(self, path: str) -> list[dict]:
    pass

class JsonTransactionLoader(TransactionLoader):
  def load(self, path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
      data = json.load(f)
    if not isinstance(data, list):
      raise ValueError("Ожидался JSON-массив записей")
    return [dict(r) for r in data if isinstance(r, dict)]


class CsvTransactionLoader(TransactionLoader):
  def load(self, path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8", newline="") as f:
      reader = csv.DictReader(f)
      return [dict(row) for row in reader]


def loader_for(path: str) -> TransactionLoader:
  # Фабрика: выбирает загрузчик по расширению
  lower = path.lower()
  if lower.endswith(".json"):
    return JsonTransactionLoader()
  if lower.endswith(".csv"):
    return CsvTransactionLoader()
  raise ValueError("Поддерживаются только .json и .csv")


# Проверяет «сырые» записи и превращает их в Operation.
class TransactionValidator:

  DATE_FORMATS = (
    "%Y-%m-%d %H:%M:%S",
    "%d/%m/%Y %H:%M",
    "%d/%m/%Y %H:%M:%S",
    "%Y-%m-%d",
  )
  
  REQUIRED_FIELDS = ("operation", "amount", "date", "status")

  # Набор допустимых операций задаётся снаружи.
  def __init__(self, valid_operations,
              account_number: str | None = None,
              account_type: str | None = None):
    self.valid_operations = frozenset(valid_operations)
    self.account_number = account_number
    self.account_type = account_type.lower() if account_type else None

  # приведение «сырых» значений
  @staticmethod
  def _normalize(raw: dict) -> dict:
    out = {}
    for key, value in raw.items():
        if isinstance(value, str):
          value = value.strip()
          if value == "" or value.lower() in ("null", "none", "nan"):
            value = None
        if key in ("amount", "balance_after") and value is not None:
          try:
            value = float(value)
          except (TypeError, ValueError):
            value = None
        if key in ("operation", "status", "account_type",
                    "account_number", "date") and value is not None:
          value = str(value).strip()
        out[key] = value
    return out

  @classmethod
  def _parse_date(cls, value):
    if not isinstance(value, str):
      return None
    for fmt in cls.DATE_FORMATS:
      try:
        return datetime.strptime(value, fmt)
      except ValueError:
        continue
    return None

  # поиск аккаунта
  def _matches_account(self, record: dict) -> bool:
    if self.account_number is not None and self.account_type is not None:
      return record.get("account_number") == self.account_number and str(record.get("account_type", "")).lower() == self.account_type
    return True

  # проверка одной записи
  def _is_valid(self, record: dict) -> bool:
    for field in self.REQUIRED_FIELDS:
      if field not in record:
        return False
    if record.get("operation") not in self.valid_operations:
      return False
    if record.get("status") not in ("success", "fail"):
      return False
    amount = record.get("amount")
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
      return False
    if amount <= 0:
      return False
    if self._parse_date(record.get("date")) is None:
      return False
    return True

  # --- публичный API ---
  def clean(self, raw_records: list[dict]) -> list[Operation]:
    cleaned: list[Operation] = []
    for raw in raw_records:
      record = self._normalize(raw)
      if not self._matches_account(record):
        continue
      if not self._is_valid(record):
        continue
      cleaned.append(Operation(
        type=record["operation"],
        amount=float(record["amount"]),
        datetime=self._parse_date(record["date"]),
        balance=record.get("balance_after") or 0.0,
        status=record["status"],
      ))
    cleaned.sort(key=lambda op: op.datetime)
    return cleaned

class Account():
  account_type = "Account"
  _account_counter = 100000 # под формат загружаемых данных из файла
  valid_operations = frozenset({"deposit", "withdraw"})

  def __init__(self, account_holder: str, balance: float = 0):
    if not self._is_valid_holder(account_holder):
      raise ValueError("Некорректное имя владельца")

    if balance < 0:
      raise ValueError("Начальный баланс не может быть отрицательным")

    self.holder = account_holder
    Account._account_counter += 1
    self.account_number = f"ACC-{Account._account_counter}"

    self._balance = float(balance)
    self._initial_balance = float(balance)
    self.operations_history = []
  
  @staticmethod
  def _is_valid_holder(name: str) -> bool:
    if not isinstance(name, str):
      return False
    # «Имя Фамилия»: первая буква заглавная, остальные строчные кирриллица или латиница.
    pattern = r"^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$"
    return re.fullmatch(pattern, name) is not None

  def _add_operation(self, op_type: str, amount: float, status: str):
    self.operations_history.append(Operation(
      type=op_type,
      amount=float(amount),
      datetime=datetime.now(),
      balance=self._balance,
      status=status,
    ))
  
  # применение загруженых и очищенных операций
  def apply_operations(self, operations: list[Operation]) -> int:
    for op in operations:
      self.operations_history.append(op)
    self._recalculate_balance()
    return len(operations)
  
  def _recalculate_balance(self):
    total = self._initial_balance
    for op in self.operations_history:
      if op.status == "success":
        if op.type in ("deposit", "interest"):
          total += op.amount
        elif op.type == "withdraw":
          total -= op.amount
      op.balance = total
    self._balance = total
  
  def deposit(self, amount: float):
    if amount <= 0:
      raise ValueError("Сумма пополнения должна быть положительной")
    self._balance += amount
    self._add_operation("deposit", amount, "success")

  def withdraw(self, amount: float) -> bool:
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    if amount > self._balance:
      self._add_operation("withdraw", amount, "fail")
      return False
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True
  
  def get_balance(self):
    return self._balance
  
  def get_history(self):
    result = []
    for op in self.operations_history:
      d = asdict(op)
      d["datetime"] = op.datetime.strftime("%Y-%m-%d %H:%M:%S")
      result.append(d)
    return result

  def analyze_operations(
    self,
    n: int = 5,
    sort_by: str = "amount",        # "amount" | "datetime" | "both"
    reverse: bool = True,           # True — по убыванию, False — по возрастанию
    min_amount = 0                  # выбрать границу для крупных операций
  ):

    if n <= 0:
      return []
    if sort_by == "amount":
      key = lambda op: op.amount
    elif sort_by == "datetime":
      key = lambda op: op.datetime
    elif sort_by == "both":
      key = lambda op: (op.amount, op.datetime)
    else:
      raise ValueError("sort_by должен быть 'amount', 'datetime' или 'both'")

    ops = [op for op in self.operations_history if op.amount >= min_amount]
    ops = sorted(ops, key=key, reverse=reverse)
    result = ops[:n]
    for op in result:
      print(f"{op.datetime:%Y-%m-%d %H:%M:%S} | "
            f"{op.type:8} | {op.amount:.2f} | "
            f"баланс: {op.balance:.2f} | {op.status}")
    return result

  def make_validator(self) -> TransactionValidator:
    return TransactionValidator(
        valid_operations=self.valid_operations,
        account_type=self.account_type,
        account_number=self.account_number,
    )

class CheckingAccount(Account):
  account_type = "checking"
  valid_operations = frozenset({"deposit", "withdraw"})


class SavingsAccount(Account):
  account_type = "savings"
  valid_operations = frozenset({"deposit", "withdraw", "interest"})

  def apply_interest(self, rate: float) -> float:
    # значение в процентах
    if rate < 0:
      raise ValueError("Процентная ставка не может быть отрицательной")
    
    interest = self._balance * rate / 100
    self._balance += interest
    if interest == 0:
      return 0.0
    
    self._add_operation("interest", interest, "success")

    return interest
  
  def withdraw(self, amount) -> bool:
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    
    limit = self.get_balance() * 0.5
    if limit < amount:
      self._add_operation("withdraw", amount, "fail")
      return False
    
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True

# Оркестратор. загрузка → очистка → применение для аккаунта
class TransactionImporter:
  def __init__(self, loader: TransactionLoader,
              validator: TransactionValidator):
    self.loader = loader
    self.validator = validator

  def import_into(self, account: Account, path: str) -> int:
    raw = self.loader.load(path)
    cleaned = self.validator.clean(raw)
    return account.apply_operations(cleaned)

# ПРОВЕРКА
checking = CheckingAccount("Иван Попов", 10000)

savings = SavingsAccount("Anna Smirnova", 1000)
loader = loader_for(JSON_PATH)
validator_s = savings.make_validator()
importer_s = TransactionImporter(loader, validator_s)
loaded_s = importer_s.import_into(savings, JSON_PATH)
print(f"[Savings] Загружено из JSON: {loaded_s}, баланс: {savings.get_balance()}")

savings.get_history()

[Savings] Загружено из JSON: 19, баланс: 4129.240000000001


[{'type': 'deposit',
  'amount': 469.0,
  'datetime': '2025-09-27 22:17:26',
  'balance': 1469.0,
  'status': 'success'},
 {'type': 'withdraw',
  'amount': 392.0,
  'datetime': '2025-09-27 22:17:26',
  'balance': 1077.0,
  'status': 'success'},
 {'type': 'deposit',
  'amount': 317.0,
  'datetime': '2025-09-28 22:17:26',
  'balance': 1394.0,
  'status': 'success'},
 {'type': 'deposit',
  'amount': 840.0,
  'datetime': '2025-10-02 22:17:26',
  'balance': 2234.0,
  'status': 'success'},
 {'type': 'interest',
  'amount': 183.28,
  'datetime': '2025-10-04 22:17:00',
  'balance': 2417.28,
  'status': 'success'},
 {'type': 'interest',
  'amount': 192.44,
  'datetime': '2025-10-06 22:17:26',
  'balance': 2609.7200000000003,
  'status': 'success'},
 {'type': 'deposit',
  'amount': 426.0,
  'datetime': '2025-10-08 22:17:26',
  'balance': 3035.7200000000003,
  'status': 'success'},
 {'type': 'deposit',
  'amount': 638.0,
  'datetime': '2025-10-09 22:17:00',
  'balance': 3673.7200000000003,
  'sta